In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn import metrics

In [2]:
data = pd.read_csv('data/data_DFT_MF.csv')
y = data[['Yield']]
X = data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])
pc_cols  = [c for c in X.columns if c.startswith("PC_MF_")]
alc_cols = [c for c in X.columns if c.startswith("Alc_MF_")]
print(f"PC columns: {len(pc_cols)}")
print(f"Alc columns: {len(alc_cols)}")

pc_pipeline = Pipeline([("pca", PCA(random_state=0))])
alc_pipeline = Pipeline([("pca", PCA(random_state=0))])
preprocessor = ColumnTransformer([("pc",  pc_pipeline,  pc_cols),("alc", alc_pipeline, alc_cols)])

pipe = Pipeline([("preprocess", preprocessor),
                 ("model", HistGradientBoostingRegressor(random_state=0,max_leaf_nodes=5,max_bins=30))])

param_grid = {"model__min_samples_leaf": [2, 3, 5],
              "model__max_depth": [4, 6],
              "model__l2_regularization": [0, 0.1, 1],
              "preprocess__pc__pca__n_components": [5, 6, 7],
              "preprocess__alc__pca__n_components": [5, 6, 7]}

r2_score_list = []
rmse_score_list = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=i)
    reg = GridSearchCV(pipe, param_grid=param_grid, cv=5, n_jobs=12)
    reg.fit(X_train, y_train["Yield"])
    best = reg.best_estimator_
    y_pred_test = best.predict(X_test)
    r2 = metrics.r2_score(y_test["Yield"], y_pred_test)
    rmse = metrics.root_mean_squared_error(y_test["Yield"], y_pred_test)
    alc_pcs = best.named_steps["preprocess"].named_transformers_["alc"].named_steps["pca"].n_components_
    pc_pcs  = best.named_steps["preprocess"].named_transformers_["pc"].named_steps["pca"].n_components_
    print(f'Run{i} R2: {r2:.2f}, RMSE: {rmse:.2f} | Alc_PCs: {alc_pcs}, PC_PCs: {pc_pcs}')
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
print('==========(Result)==========')
print(f'Mean R2:  {np.mean(r2_score_list):.3f}')
print(f'SD R2:    {np.std(r2_score_list):.3f}')
print(f'Mean RMSE:{np.mean(rmse_score_list):.3f}')
print(f'SD RMSE:  {np.std(rmse_score_list):.3f}')

PC columns: 102
Alc columns: 108
Run0 R2: 0.57, RMSE: 17.61 | Alc_PCs: 5, PC_PCs: 6
Run1 R2: 0.44, RMSE: 15.67 | Alc_PCs: 5, PC_PCs: 7
Run2 R2: 0.81, RMSE: 8.07 | Alc_PCs: 6, PC_PCs: 7
Run3 R2: 0.40, RMSE: 20.60 | Alc_PCs: 5, PC_PCs: 7
Run4 R2: 0.76, RMSE: 12.07 | Alc_PCs: 5, PC_PCs: 7
Run5 R2: 0.55, RMSE: 14.01 | Alc_PCs: 5, PC_PCs: 7
Run6 R2: 0.60, RMSE: 12.58 | Alc_PCs: 5, PC_PCs: 6
Run7 R2: 0.33, RMSE: 16.93 | Alc_PCs: 5, PC_PCs: 7
Run8 R2: 0.67, RMSE: 12.87 | Alc_PCs: 5, PC_PCs: 7
Run9 R2: 0.43, RMSE: 15.06 | Alc_PCs: 6, PC_PCs: 7
==========(Result)==========
Mean R2:  0.556
SD R2:    0.150
Mean RMSE:14.546
SD RMSE:  3.291


In [3]:
data = pd.read_csv('data/data_DFT_RDKit.csv')
y = data[['Yield']]
X = data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])

pc_cols  = [c for c in X.columns if c.startswith("RDKit_PC_")]
alc_cols = [c for c in X.columns if c.startswith("RDKit_Alc_")]
print(f"PC columns: {len(pc_cols)}")
print(f"Alc columns: {len(alc_cols)}")

pc_pipeline = Pipeline([("scaler", StandardScaler()),
                        ("pca", PCA(random_state=0))])
alc_pipeline = Pipeline([("scaler", StandardScaler()),
                         ("pca", PCA(random_state=0))])
preprocessor = ColumnTransformer([("pc",  pc_pipeline,  pc_cols),("alc", alc_pipeline, alc_cols)])

pipe = Pipeline([("preprocess", preprocessor),
                 ("model", HistGradientBoostingRegressor(random_state=0,max_leaf_nodes=5,max_bins=30))])

param_grid = {"model__min_samples_leaf": [2, 3, 5],
              "model__max_depth": [4, 6],
              "model__l2_regularization": [0, 0.1, 1],
              "preprocess__pc__pca__n_components": [5, 6, 7],
              "preprocess__alc__pca__n_components": [5, 6, 7]}

r2_score_list = []
rmse_score_list = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=i)
    reg = GridSearchCV(pipe, param_grid=param_grid, cv=5, n_jobs=12)
    reg.fit(X_train, y_train["Yield"])
    best = reg.best_estimator_
    y_pred_test = best.predict(X_test)
    r2 = metrics.r2_score(y_test["Yield"], y_pred_test)
    rmse = metrics.root_mean_squared_error(y_test["Yield"], y_pred_test)
    alc_pcs = best.named_steps["preprocess"].named_transformers_["alc"].named_steps["pca"].n_components_
    pc_pcs  = best.named_steps["preprocess"].named_transformers_["pc"].named_steps["pca"].n_components_
    print(f'Run{i} R2: {r2:.2f}, RMSE: {rmse:.2f} | Alc_PCs: {alc_pcs}, PC_PCs: {pc_pcs}')
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
print('==========(Result)==========')
print(f'Mean R2:  {np.mean(r2_score_list):.3f}')
print(f'SD R2:    {np.std(r2_score_list):.3f}')
print(f'Mean RMSE:{np.mean(rmse_score_list):.3f}')
print(f'SD RMSE:  {np.std(rmse_score_list):.3f}')

PC columns: 129
Alc columns: 118
Run0 R2: 0.61, RMSE: 16.75 | Alc_PCs: 5, PC_PCs: 7
Run1 R2: 0.47, RMSE: 15.27 | Alc_PCs: 6, PC_PCs: 6
Run2 R2: 0.73, RMSE: 9.66 | Alc_PCs: 7, PC_PCs: 7
Run3 R2: 0.32, RMSE: 22.04 | Alc_PCs: 7, PC_PCs: 6
Run4 R2: 0.59, RMSE: 15.58 | Alc_PCs: 7, PC_PCs: 7
Run5 R2: 0.51, RMSE: 14.75 | Alc_PCs: 6, PC_PCs: 7
Run6 R2: 0.62, RMSE: 12.22 | Alc_PCs: 6, PC_PCs: 5
Run7 R2: 0.52, RMSE: 14.38 | Alc_PCs: 5, PC_PCs: 5
Run8 R2: 0.33, RMSE: 18.25 | Alc_PCs: 7, PC_PCs: 5
Run9 R2: 0.50, RMSE: 14.10 | Alc_PCs: 7, PC_PCs: 6
==========(Result)==========
Mean R2:  0.520
SD R2:    0.121
Mean RMSE:15.300
SD RMSE:  3.164


In [4]:
data = pd.read_csv('data/data_DFT_mordred.csv')
y = data[['Yield']]
X = data.drop(columns=['PC_ID', 'Alc_ID', 'Yield', 'PC_SMILES', 'Alc_SMILES'])

pc_cols  = [c for c in X.columns if c.startswith("mordred_PC_")]
alc_cols = [c for c in X.columns if c.startswith("mordred_Alc_")]
print(f"PC columns: {len(pc_cols)}")
print(f"Alc columns: {len(alc_cols)}")

pc_pipeline = Pipeline([("scaler", StandardScaler()),
                        ("pca", PCA(random_state=0))])
alc_pipeline = Pipeline([("scaler", StandardScaler()),
                         ("pca", PCA(random_state=0))])
preprocessor = ColumnTransformer([("pc",  pc_pipeline,  pc_cols),("alc", alc_pipeline, alc_cols)])

pipe = Pipeline([("preprocess", preprocessor),
                 ("model", HistGradientBoostingRegressor(random_state=0,max_leaf_nodes=5,max_bins=30))])

param_grid = {"model__min_samples_leaf": [2, 3, 5],
              "model__max_depth": [4, 6],
              "model__l2_regularization": [0, 0.1, 1],
              "preprocess__pc__pca__n_components": [5, 6, 7],
              "preprocess__alc__pca__n_components": [5, 6, 7]}

r2_score_list = []
rmse_score_list = []
for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=i)
    reg = GridSearchCV(pipe, param_grid=param_grid, cv=5, n_jobs=12)
    reg.fit(X_train, y_train["Yield"])
    best = reg.best_estimator_
    y_pred_test = best.predict(X_test)
    r2 = metrics.r2_score(y_test["Yield"], y_pred_test)
    rmse = metrics.root_mean_squared_error(y_test["Yield"], y_pred_test)
    alc_pcs = best.named_steps["preprocess"].named_transformers_["alc"].named_steps["pca"].n_components_
    pc_pcs  = best.named_steps["preprocess"].named_transformers_["pc"].named_steps["pca"].n_components_
    print(f'Run{i} R2: {r2:.2f}, RMSE: {rmse:.2f} | Alc_PCs: {alc_pcs}, PC_PCs: {pc_pcs}')
    r2_score_list.append(r2)
    rmse_score_list.append(rmse)
print('==========(Result)==========')
print(f'Mean R2:  {np.mean(r2_score_list):.3f}')
print(f'SD R2:    {np.std(r2_score_list):.3f}')
print(f'Mean RMSE:{np.mean(rmse_score_list):.3f}')
print(f'SD RMSE:  {np.std(rmse_score_list):.3f}')

PC columns: 1153
Alc columns: 1145
Run0 R2: 0.68, RMSE: 15.37 | Alc_PCs: 6, PC_PCs: 7
Run1 R2: 0.69, RMSE: 11.64 | Alc_PCs: 7, PC_PCs: 5
Run2 R2: 0.65, RMSE: 10.99 | Alc_PCs: 7, PC_PCs: 6
Run3 R2: 0.70, RMSE: 14.65 | Alc_PCs: 6, PC_PCs: 7
Run4 R2: 0.76, RMSE: 12.02 | Alc_PCs: 7, PC_PCs: 6
Run5 R2: 0.56, RMSE: 13.94 | Alc_PCs: 6, PC_PCs: 5
Run6 R2: 0.28, RMSE: 16.80 | Alc_PCs: 7, PC_PCs: 7
Run7 R2: 0.56, RMSE: 13.63 | Alc_PCs: 6, PC_PCs: 6
Run8 R2: 0.50, RMSE: 15.83 | Alc_PCs: 6, PC_PCs: 6
Run9 R2: 0.65, RMSE: 11.76 | Alc_PCs: 7, PC_PCs: 7
==========(Result)==========
Mean R2:  0.603
SD R2:    0.130
Mean RMSE:13.661
SD RMSE:  1.899
